In [ ]:
import cv2 
import numpy as np
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

In [ ]:
IMG_DIR = "/kaggle/input/datasets/ababeiraul/findtheduck/custom_archive/images"

TRAIN_IMG_DIR = f"{IMG_DIR}/train_dataset"

CSV_DIR = "/kaggle/input/datasets/ababeiraul/findtheduck"

train_df = pd.read_csv(f"{CSV_DIR}/train_data.csv")

print(train_df.head(3))

In [ ]:
ducks_df = train_df[train_df["DuckOrNoDuck"] == 1].head(20)

print(ducks_df.head(5))

In [ ]:
import os

for idx, row in ducks_df.iterrows():
    img_id = row["DatapointID"]
    true_pixels = row["PixelCount"]

    x1, y1, x2, y2 = map(int, str(row["BoundingBox"]).split())

    img_path = os.path.join(TRAIN_IMG_DIR, f"{img_id}.png")
    img = cv2.imread(img_path)

    crop = img[y1 : y2 + 1, x1 : x2  + 1]

    black_0 = np.sum(np.all(crop == 0, axis = -1))
    black_5 = np.sum(np.all(crop <= 5, axis = -1))
    black_15 = np.sum(np.all(crop <= 15, axis = -1))

    print(
        f"ID={img_id} | Real PixelCount={true_pixels} | "
        f"RGB==0: {black_0} | <=5: {black_5} | <=15: {black_15}"
    )
    

In [ ]:
real_areas = []
real_densities = []
real_aspects = []
real_pads = []

for _, row in train_df[train_df["DuckOrNoDuck"] == 1].iterrows():
    img_path = os.path.join(TRAIN_IMG_DIR, f"{row['DatapointID']}.png")
    img = cv2.imread(img_path)
    tx1, ty1, tx2, ty2 = map(int, str(row["BoundingBox"]).split())

    crop = img[ty1:ty2+1, tx1:tx2+1]
    mask0 = cv2.inRange(crop, (0, 0, 0), (0, 0, 0))
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask0, connectivity=8)

    if num_labels <= 1:
        continue
        
    best_id = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    area = int(stats[best_id, cv2.CC_STAT_AREA])
    bx1 = int(stats[best_id, cv2.CC_STAT_LEFT])
    by1 = int(stats[best_id, cv2.CC_STAT_TOP])
    bw = int(stats[best_id, cv2.CC_STAT_WIDTH])
    bh = int(stats[best_id, cv2.CC_STAT_HEIGHT])
    bx2 = bx1 + bw - 1
    by2 = by1 + bh - 1

    real_areas.append(area)
    real_densities.append(area / max(1, bw * bh))
    real_aspects.append(max(bw, bh) / max(1, min(bw, bh)))
    real_pads.extend([bx1, by1, (tx2 - tx1) - bx2, (ty2 - ty1) - by2])

print(f"Total rațe cu nucleu (0,0,0) găsite: {len(real_areas)} / {(train_df['DuckOrNoDuck']==1).sum()}")
print(f"AREA      -> min: {np.min(real_areas)}, p1: {np.percentile(real_areas, 1):.1f}, max: {np.max(real_areas)}")
print(f"DENSITY   -> min: {np.min(real_densities):.3f}, p99: {np.percentile(real_densities, 99):.3f}, max: {np.max(real_densities):.3f}")
print(f"ASPECT    -> min: {np.min(real_aspects):.2f}, p99: {np.percentile(real_aspects, 99):.2f}, max: {np.max(real_aspects):.2f}")
print(f"PAD REAL  -> mediană: {np.median(real_pads):.1f}, medie: {np.mean(real_pads):.2f} pixeli")

In [ ]:
def find_duck_bbox(img_path, dark_thresh=50, search_margin=40):
    """
    return (has_duck, (x1, y1, x2, y2), (H, W)) for a given img_path
    """


    img = cv2.imread(img_path)
    if img is None:
        raise FileNotFoundError(f"Nu găsesc imaginea: {img_path}")


    H, W = img.shape[: 2]

    # a mask for the pure black pixels (0, 0, 0)
    mask0 = cv2.inRange(img, (0, 0, 0), (0, 0, 0))

    # componente conexe and then iterate and remove them
    # connectivity = 8 so we can connect also on the diagonals
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask0, connectivity = 8)

    # labels is a maxtrix of W * H spaces
    # for each space -> 0 if background
    #                -> id if connectedComponent
    
    comps = []
    for label_id in range(1, num_labels):
        area = int(stats[label_id, cv2.CC_STAT_AREA])

        w = int(stats[label_id, cv2.CC_STAT_WIDTH])
        h = int(stats[label_id, cv2.CC_STAT_HEIGHT])

        density = area / max(1, w * h)
        aspect = max(w, h) / max(1, min(w, h))

        if 12 <= area <= 7000 and 0.02 <= density <= 0.70 and aspect <= 7.5:
            x1 = int(stats[label_id, cv2.CC_STAT_LEFT])
            y1 = int(stats[label_id, cv2.CC_STAT_TOP])
            x2 = x1 + w - 1
            y2 = y1 + h - 1
            
            comps.append((area, density, (x1, y1, x2, y2)))

    if not comps:
        return 0, None, (H, W)

    comps.sort(key = lambda x: x[0], reverse = True)
    x1, y1, x2, y2 = comps[0][2]

    rx1 = max(0, x1 - search_margin)
    ry1 = max(0, y1 - search_margin)
    rx2 = min(W - 1, x2 + search_margin)
    ry2 = min(H - 1, y2 + search_margin)

    roi = img[ry1:ry2+1, rx1:rx2+1]

    dark_mask = cv2.inRange(roi, (0, 0, 0), (dark_thresh, dark_thresh, dark_thresh))
    n_roi, lbl_roi, st_roi, _ = cv2.connectedComponentsWithStats(dark_mask, connectivity=8)

    core_mask = cv2.inRange(roi, (0, 0, 0), (0, 0, 0))
    overlapping_labels = np.unique(lbl_roi[core_mask > 0])
    overlapping_labels = overlapping_labels[overlapping_labels > 0]

    if len(overlapping_labels) > 0:
        ys, xs = np.where(np.isin(lbl_roi, overlapping_labels))
        fx1 = rx1 + int(xs.min())
        fy1 = ry1 + int(ys.min())
        fx2 = rx1 + int(xs.max())
        fy2 = ry1 + int(ys.max())
        return 1, (fx1, fy1, fx2, fy2), (H, W)

    return 1, (x1, y1, x2, y2), (H, W)

    

In [ ]:
def compute_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])
    inter = max(0, xB - xA + 1) * max(0, yB - yA + 1)
    areaA = (boxA[2] - boxA[0] + 1) * (boxA[3] - boxA[1] + 1)
    areaB = (boxB[2] - boxB[0] + 1) * (boxB[3] - boxB[1] + 1)
    return inter / float(areaA + areaB - inter)

In [ ]:
paths = [os.path.join(TRAIN_IMG_DIR, f"{dp_id}.png") for dp_id in train_df["DatapointID"]]

In [ ]:
with ThreadPoolExecutor(max_workers=8) as ex:
    results = list(tqdm(ex.map(find_duck_bbox, paths), total=len(paths)))

In [ ]:
cls_correct = 0
ious = []

for (_, row), (pred_cls, box, _) in zip(train_df.iterrows(), results):

    if pred_cls == row["DuckOrNoDuck"]:
        cls_correct += 1

    if row["DuckOrNoDuck"] == 1 and box is not None:
        # true values
        tx1, ty1, tx2, ty2 = map(int, str(row["BoundingBox"]).split())
        ious.append(compute_iou(box, (tx1, ty1, tx2, ty2)))       

print(f"Acuratețe Subtask 1 pe TOT train: {cls_correct / len(train_df):.4f}")
print(f"IoU mediu Subtask 3 pe TOT train: {np.mean(ious):.4f}")

In [ ]:
TEST_IMG_DIR = f"{IMG_DIR}/test_dataset"

test_files = [f for f in os.listdir(TEST_IMG_DIR) if f.endswith(".png")]
test_ids = sorted([int(os.path.splitext(f)[0]) for f in test_files])

print(f"Total imagini de test găsite: {len(test_ids)} (de la {test_ids[0]} la {test_ids[-1]})")

test_paths = [os.path.join(TEST_IMG_DIR, f"{dp_id}.png") for dp_id in test_ids]

In [ ]:
with ThreadPoolExecutor(max_workers=8) as ex:
    test_results = list(tqdm(ex.map(find_duck_bbox, test_paths), total=len(test_paths)))

In [ ]:
rows_t1 = []
rows_t2 = []
rows_t3 = []

for dp_id, (has_duck, box, _) in zip(test_ids, test_results):
    rows_t1.append({"subtaskID": 1, "datapointID": dp_id, "answer": int(has_duck)})
    rows_t2.append({"subtaskID": 2, "datapointID": dp_id, "answer": 0})  # placeholder momentan
    if has_duck == 1 and box is not None:
        bbox_str = f"{box[0]} {box[1]} {box[2]} {box[3]}"
    else:
        bbox_str = "0 0 0 0"
    rows_t3.append({"subtaskID": 3, "datapointID": dp_id, "answer": bbox_str})

# 5. Concatenăm Subtask 1 + Subtask 2 + Subtask 3 și salvăm CSV-ul final
submission = pd.concat(
    [pd.DataFrame(rows_t1), pd.DataFrame(rows_t2), pd.DataFrame(rows_t3)],
    ignore_index=True
)

out_file = "/kaggle/working/submission_t1_t3.csv"
submission.to_csv(out_file, index=False)

print(f"\nGata! Salvat {out_file} | Shape: {submission.shape} | Nulls: {submission.isna().sum().sum()}")
print(submission.head(5))
print(submission.tail(5))